# Titanic Dataset Prediction

Load the training and test data.

In [1]:
import pandas as pd
import numpy as np
train_path = "train.csv"
test_path = "test.csv"

data= pd.read_csv(train_path)
test = pd.read_csv(test_path)

PassengerID=test["PassengerId"]

## Data Cleaning & Feature Engineering

Clean missing values, extract titles, and encode categorical features.

In [2]:
def data_cleaning(a):
  a=a.drop(["PassengerId","Ticket","Cabin"],axis=1) 

  cols_with_missing_values=["SibSp","Parch","Fare","Age"]
  for i in cols_with_missing_values:
    a[i].fillna(a[i].median(),inplace=True)         
  a["Embarked"].fillna("U",inplace=True)
  a["Title"]=a["Name"].str.extract('([A-Za-z]+)\\.',expand=False)
  a=a.drop(["Name"],axis=1)
  return a

data=data_cleaning(data)
test=data_cleaning(test)

for dataset in [data, test]:
    dataset["Title"] = dataset["Title"].replace(["Mlle", "Ms"], "Miss")
    dataset["Title"] = dataset["Title"].replace("Mme", "Mrs")
    dataset["Title"] = dataset["Title"].replace(
        ["Countess", "Lady", "Sir", "Jonkheer", "Don", "Dona", "Dr", "Rev", "Major", "Col", "Capt"],
        "Rare"
    )


from sklearn.preprocessing import LabelEncoder    
for x in ["Sex","Embarked","Title"]:
  lbl=LabelEncoder()
  lbl.fit(pd.concat([data[x],test[x]]))
  data[x]=lbl.transform(data[x])
  test[x]=lbl.transform(test[x])
  print(lbl.classes_)


data["FamilySize"] = data["SibSp"] + data["Parch"] + 1 
test["FamilySize"] = test["SibSp"] + test["Parch"] + 1

data["IsAlone"] = (data["FamilySize"] == 1).astype(int)
test["IsAlone"] = (test["FamilySize"] == 1).astype(int)

data["AgeGroup"] = pd.cut(data["Age"], bins=[0,12,18,30,50,80], labels=[0,1,2,3,4])  
test["AgeGroup"] = pd.cut(test["Age"], bins=[0,12,18,30,50,80], labels=[0,1,2,3,4])

data["AgeGroup"] = data["AgeGroup"].astype(int)
test["AgeGroup"] = test["AgeGroup"].astype(int)

['female' 'male']
['C' 'Q' 'S' 'U']
['Master' 'Miss' 'Mr' 'Mrs' 'Rare']


## Model Preparation

Split data, scale features, and prepare training/validation sets.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

y=data["Survived"]
x=data.drop(["Survived"],axis=1)

x_train,x_val,y_train,y_val=train_test_split(x,y,test_size=0.2,random_state=128)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
test_scaled = scaler.transform(test)

## Random Forest Model & Hyperparameter Search

Train a random forest, evaluate it, and run a grid search for best params.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(x_train, y_train)
rf_preds = rf.predict(x_val)
print("Random Forest Accuracy:", accuracy_score(y_val, rf_preds))
from sklearn.model_selection import GridSearchCV

params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8, None],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid=params, cv=5, scoring="accuracy")
grid.fit(x_train, y_train)
print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)



Random Forest Accuracy: 0.8100558659217877
Best Parameters: {'max_depth': 4, 'min_samples_split': 2, 'n_estimators': 100}
Best Accuracy: 0.8244262779474048


## Random Forest (Scaled) Evaluation

Train and evaluate a random forest on scaled features.

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(
    n_estimators=200,       
    max_depth=6,            
    random_state=0
)

rf.fit(x_train_scaled, y_train)
rf_preds = rf.predict(x_val_scaled)

print("🌲 Random Forest Accuracy:", accuracy_score(y_val, rf_preds))


🌲 Random Forest Accuracy: 0.8603351955307262


## Submission 

Create submission file using Random Forest predictions.

In [8]:
test_predictions = rf.predict(test_scaled)
submission_df = pd.DataFrame({'PassengerId': PassengerID, 'Survived': test_predictions})
submission_df.to_csv('submission.csv', index=False)
print("Submission file created successfully!")

Submission file created successfully!
